In [ ]:
import os
import datetime
import warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers
from sklearn.metrics import mean_absolute_error
from models import create_arcface_model
from utils import loss_val_graph

warnings.filterwarnings('ignore')
print('TensorFlow version:', tf.__version__)

In [ ]:
train_ds = tf.data.Dataset.load(r'data/ds/train_ds') \
    .cache().shuffle(buffer_size=1000, seed=42).prefetch(buffer_size=tf.data.AUTOTUNE)

# ✅ valid_ds TIDAK di-shuffle — agar metrik evaluasi stabil setiap epoch
valid_ds = tf.data.Dataset.load(r'data/ds/val_ds') \
    .cache().prefetch(buffer_size=tf.data.AUTOTUNE)

train_ds, valid_ds

Load Arcface

In [ ]:
# ── [Cell 3] Verifikasi: Load ArcFace pre-trained weights ──────────────────
# Cell ini hanya untuk memastikan create_arcface_model() berjalan dengan benar.
# arcface_112 yang dibuat di sini akan DIGUNAKAN LANGSUNG di Cell 4.
arcface_112 = create_arcface_model(load_weights=True)
arcface_112.trainable = False

print(f'\nArcFace output shape: {arcface_112.output_shape}')  # Harusnya (None, 512)
print(f'Jumlah layer ArcFace : {len(arcface_112.layers)}')
print(f'Trainable            : {arcface_112.trainable}')

BUILD MODEL

In [ ]:
# ── [Cell 4] Build Model ────────────────────────────────────────────────────
# Arsitektur Information Bottleneck (mengecil): 256 → 128
# L2 Regularizer ringan (1e-4) — tidak memicu Dying ReLU
# Referensi: Tishby & Zaslavsky (2015), Zhang et al. (2017)

inputs = keras.layers.Input(shape=(10, 112, 112, 3), name='Input_Sequence')

# Rescale [0,255] → [0,1]
x = keras.layers.TimeDistributed(keras.layers.Rescaling(scale=1./255.0), name='Rescaling')(inputs)

# Ekstrak embedding per-frame dengan ArcFace → (batch, 10, 512)
x = keras.layers.TimeDistributed(arcface_112, name='ArcFace_Embeddings')(x)

# Temporal modelling
x = keras.layers.LSTM(128, return_sequences=True,
                        recurrent_dropout=0.2, name='LSTM_1')(x)
x = keras.layers.LSTM(64, recurrent_dropout=0.2, name='LSTM_2')(x)
x = keras.layers.Dropout(0.2, name='Dropout_LSTM')(x)

# Dense head — arsitektur bottleneck: 256 → 128
x = keras.layers.Dense(256, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4), name='Dense_1')(x)
x = keras.layers.Dropout(0.3, name='Dropout_1')(x)
x = keras.layers.Dense(128, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4), name='Dense_2')(x)
x = keras.layers.Dropout(0.5, name='Dropout_2')(x)

# Output: 5 trait OCEAN, nilai antara 0–1
x = keras.layers.Dense(5, activation='sigmoid', name='OCEAN_Output')(x)

model = keras.models.Model(inputs=inputs, outputs=x,
                           name='ArcFace_OCEAN_Predictor')
keras.utils.plot_model(model, show_shapes=True)

COMPILE MODEL

In [ ]:
t = datetime.datetime.now().strftime('%m%d_%H%M%S')

# optimizer = keras.optimizers.Adam(learning_rate=1e-3)             
# OPTIMIZER_NAME = 'adam'
optimizer = keras.optimizers.RMSprop(learning_rate=1e-3)     
OPTIMIZER_NAME = 'rmsprop'
# optimizer = keras.optimizers.SGD(learning_rate=1e-2, momentum=0.9)
# OPTIMIZER_NAME = 'sgdm'

os.makedirs(f'./weights/faces/{t}', exist_ok=True)
weight_path = f'./weights/faces/{t}/face.h5'
csv_path    = f'./weights/faces/{t}/history.csv'

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=1, restore_best_weights=True)
check_point = keras.callbacks.ModelCheckpoint(filepath=weight_path, monitor='val_mae', mode='min', save_best_only=True, save_weights_only=True, verbose=1)
csv_logger = keras.callbacks.CSVLogger(csv_path, append=False)

model.compile(loss='mse', optimizer=optimizer, metrics=['mae'])
print(f'Optimizer       : {OPTIMIZER_NAME}')
print(f'Weights path    : {weight_path}')
print(f'CSV history     : {csv_path}')

TRAINING MODEL

In [ ]:
history = model.fit(train_ds, validation_data=valid_ds, batch_size=8, epochs=100, callbacks=[early_stopping, check_point, csv_logger])
print(f'\n[OK] Training selesai. Bobot terbaik disimpan di: {weight_path}')

### Simpan Full Model (untuk Deployment Flask)

In [ ]:
# ── [Cell 8] Simpan Full Model ─────────────────────────────────────────────
# Simpan full model .h5 agar bisa langsung di-load Flask
full_model_path = f'./weights/faces/{t}/face_full_model.h5'
model.save(full_model_path)
print(f'[OK] Full model tersimpan → {full_model_path}')

### Evaluasi Model

In [ ]:
train_ds = tf.data.Dataset.load(r'data/ds/train_ds')
loss, mae = model.evaluate(train_ds)
(1-mae)*100

In [ ]:
y_true = np.concatenate([y for x,y in train_ds])
y_pred = model.predict(train_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

In [ ]:
val_ds = tf.data.Dataset.load(r'data/ds/val_ds')
loss, mae = model.evaluate(val_ds)
(1-mae)*100

In [ ]:
y_true = np.concatenate([y for x,y in val_ds])
y_pred = model.predict(val_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

In [ ]:
test_ds = tf.data.Dataset.load(r'data/ds/test_ds')
loss, mae = model.evaluate(test_ds)
(1-mae)*100

In [ ]:
y_true = np.concatenate([y for x,y in test_ds])
y_pred = model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

PLOT TRAINING CURVE

In [ ]:
# ── [Cell 7] Plot Training Curve ───────────────────────────────────────────
plot_path = f'./weights/faces/{t}/training_curve.png'
loss_val_graph(history, save_path=plot_path)
print(f'[OK] Plot disimpan → {plot_path}')